# Pandas

This notebook is a focused study file for the **pandas** Python library.

## Learning Goals

- Understand pandas objects: `Series`, `DataFrame`, `Index`, and dtypes.
- Load, inspect, clean, filter, aggregate, join, reshape, and export tabular data.
- Write readable pandas workflows using method chaining.
- Apply advanced pandas features such as categorical data, time series operations, window functions, and performance-aware patterns.

## 1. Import Pandas

The standard alias for pandas is `pd`. This alias is used in almost every pandas codebase.

In [1]:
import pandas as pd

# Display wider tables in notebooks.
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

pd.__version__

'3.0.1'

## 2. Core Pandas Objects

### Series

A `Series` is a one-dimensional labeled array.

### DataFrame

A `DataFrame` is a two-dimensional table made of aligned `Series` objects.

### Index

An `Index` labels rows or columns. Good index design can make selection, alignment, joins, and time series work cleaner.

In [2]:
# A Series with explicit labels.
sales_series = pd.Series(
    [1200, 1550, 980],
    index=["North", "South", "West"],
    name="monthly_sales"
)

# A DataFrame from a dictionary of equal-length lists.
employees = pd.DataFrame({
    "employee_id": [101, 102, 103, 104, 105, 106],
    "name": ["Asha", "Ben", "Chitra", "Dev", "Elena", "Farah"],
    "department": ["Sales", "Sales", "Tech", "Tech", "HR", "Tech"],
    "city": ["Delhi", "Mumbai", "Bengaluru", "Bengaluru", "Delhi", "Hyderabad"],
    "salary": [65000, 72000, 95000, 105000, 58000, 88000],
    "experience_years": [3, 5, 6, 8, 2, 4]
})

sales_series, employees

(North    1200
 South    1550
 West      980
 Name: monthly_sales, dtype: int64,
    employee_id    name department       city  salary  experience_years
 0          101    Asha      Sales      Delhi   65000                 3
 1          102     Ben      Sales     Mumbai   72000                 5
 2          103  Chitra       Tech  Bengaluru   95000                 6
 3          104     Dev       Tech  Bengaluru  105000                 8
 4          105   Elena         HR      Delhi   58000                 2
 5          106   Farah       Tech  Hyderabad   88000                 4)

In [3]:
series = pd.Series(
    ["English Learning", "DSA", "AI/ML", "Stay calm"],
    index = [1,2,3,4]
)

print(series)

1    English Learning
2                 DSA
3               AI/ML
4           Stay calm
dtype: str


## 3. Inspecting Data

Before cleaning or modeling data, inspect its structure.

Common inspection methods:

- `head()` and `tail()` preview rows.
- `shape` gives row and column count.
- `info()` summarizes columns, non-null values, and dtypes.
- `describe()` gives summary statistics.
- `value_counts()` shows category frequencies.

In [4]:
# Preview the first rows.
employees.head()

# Check dimensions.
employees.shape

print("_" * 50)

# Summary of column dtypes and missing values.
employees.info()
print("_" * 50)

# Numeric summary statistics.
employees.describe()
print("_" * 50)

# Frequency count for a categorical column.
employees["department"].value_counts()

__________________________________________________
<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   employee_id       6 non-null      int64
 1   name              6 non-null      str  
 2   department        6 non-null      str  
 3   city              6 non-null      str  
 4   salary            6 non-null      int64
 5   experience_years  6 non-null      int64
dtypes: int64(3), str(3)
memory usage: 513.0 bytes
__________________________________________________
__________________________________________________


department
Tech     3
Sales    2
HR       1
Name: count, dtype: int64

## 4. Selection: Columns, Rows, and Conditions

Pandas has two important explicit indexers:

- `loc`: label-based selection.
- `iloc`: integer-position-based selection.

For production work, prefer `loc` and `iloc` because they make intent clear.

In [5]:
employees

,employee_id,name,department,city,salary,experience_years
0,101,Asha,Sales,Delhi,65000,3
1,102,Ben,Sales,Mumbai,72000,5
2,103,Chitra,Tech,Bengaluru,95000,6
3,104,Dev,Tech,Bengaluru,105000,8
4,105,Elena,HR,Delhi,58000,2
5,106,Farah,Tech,Hyderabad,88000,4


In [6]:
# Select one column as a Series.
df = employees["salary"]
print(df)
print("_" * 50)

# Select multiple columns as a DataFrame.
df = employees[["name", "department", "salary"]]
print(df)
print("_" * 50)

# Label-based row and column selection. 
df = employees.loc[0:3, ["name", "department", "salary"]] # 0:3 includes rows 0, 1, 2 and 3 and the columns specified.
print(df)
print("_" * 50)

# Position-based row and column selection.
df = employees.iloc[0:3, 1:4]   # 0:3 includes rows 0, 1, 2 and 1:4 includes columns 1, 2 and 3.
print(df)
print("_" * 50)

# Boolean filtering: employees in Tech with salary above 90000.
high_paid_tech = employees.loc[
    (employees["department"] == "Tech") & (employees["salary"] > 90000),
    ["name", "city", "salary", "experience_years"]
]

print(high_paid_tech)

0     65000
1     72000
2     95000
3    105000
4     58000
5     88000
Name: salary, dtype: int64
__________________________________________________
     name department  salary
0    Asha      Sales   65000
1     Ben      Sales   72000
2  Chitra       Tech   95000
3     Dev       Tech  105000
4   Elena         HR   58000
5   Farah       Tech   88000
__________________________________________________
     name department  salary
0    Asha      Sales   65000
1     Ben      Sales   72000
2  Chitra       Tech   95000
3     Dev       Tech  105000
__________________________________________________
     name department       city
0    Asha      Sales      Delhi
1     Ben      Sales     Mumbai
2  Chitra       Tech  Bengaluru
__________________________________________________
     name       city  salary  experience_years
2  Chitra  Bengaluru   95000                 6
3     Dev  Bengaluru  105000                 8


## 5. Creating and Transforming Columns

New columns are often created from existing columns. Use vectorized operations instead of row-by-row loops when possible.

Vectorized pandas code is usually shorter, clearer, and faster.

In [7]:
employees_enriched = employees.copy()
print(employees_enriched)
print("_" * 90)

# Create a salary band using pd.cut.
employees_enriched["salary_band"] = pd.cut(
    employees_enriched["salary"],
    bins=[0, 70000, 90000, 120000],   # Define the bin edges.
    labels=["low", "medium", "high"]  # Low : 0-70000, Medium : 70000-90000, High : 90000-120000
)
print(employees_enriched)
print("_" * 90)

# Create a derived metric.
employees_enriched["salary_per_year_exp"] = (
    employees_enriched["salary"] / employees_enriched["experience_years"]
).round(2)  # Round to 2 decimal places.   
print(employees_enriched)
print("_" * 90)

# Map department names to shorter codes.
department_codes = {"Sales": "SLS", "Tech": "TEC", "HR": "HUM"}
employees_enriched["department_code"] = employees_enriched["department"].map(department_codes)
print(employees_enriched)

   employee_id    name department       city  salary  experience_years
0          101    Asha      Sales      Delhi   65000                 3
1          102     Ben      Sales     Mumbai   72000                 5
2          103  Chitra       Tech  Bengaluru   95000                 6
3          104     Dev       Tech  Bengaluru  105000                 8
4          105   Elena         HR      Delhi   58000                 2
5          106   Farah       Tech  Hyderabad   88000                 4
__________________________________________________________________________________________
   employee_id    name department       city  salary  experience_years salary_band
0          101    Asha      Sales      Delhi   65000                 3         low
1          102     Ben      Sales     Mumbai   72000                 5      medium
2          103  Chitra       Tech  Bengaluru   95000                 6        high
3          104     Dev       Tech  Bengaluru  105000                 8        hi

## 6. Missing Data

Real datasets usually contain missing values. Pandas represents missing values as `NaN`, `NaT`, or `pd.NA` depending on dtype.

Common methods:

- `isna()` detects missing values.
- `dropna()` removes missing rows or columns.
- `fillna()` fills missing values.
- `interpolate()` fills numeric or time series gaps.

In [15]:
raw_customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5],
    "age": [25, None, 41, 36, None],
    "city": ["Delhi", "Mumbai", None, "Pune", "Delhi"],
    "spend": [1200, 850, None, 2200, 1750]
})
print(raw_customers)
print("_" * 50)

# Count missing values per column.
missing_report = raw_customers.isna().sum()
print(missing_report)
print("_"*50)

# Fill age with median, city with a placeholder, and spend with 0.
customers_clean = raw_customers.assign(
    age=raw_customers["age"].fillna(raw_customers["age"].median()),
    city=raw_customers["city"].fillna("Unknown"),
    spend=raw_customers["spend"].fillna(0)
)
print(customers_clean)

   customer_id   age    city   spend
0            1  25.0   Delhi  1200.0
1            2   NaN  Mumbai   850.0
2            3  41.0     NaN     NaN
3            4  36.0    Pune  2200.0
4            5   NaN   Delhi  1750.0
__________________________________________________
customer_id    0
age            2
city           1
spend          1
dtype: int64
__________________________________________________
   customer_id   age     city   spend
0            1  25.0    Delhi  1200.0
1            2  36.0   Mumbai   850.0
2            3  41.0  Unknown     0.0
3            4  36.0     Pune  2200.0
4            5  36.0    Delhi  1750.0


## 7. Sorting and Ranking

Sorting changes display or processing order. Ranking assigns relative positions based on values.

Ranking is useful for leaderboards, top-N reports, and percentile-style analysis.

In [21]:
# Sort by department first, then salary descending within department.
sorted_employees = employees.sort_values(
    by=["department", "salary"],
    ascending=[True, False]
)
print(sorted_employees)
print("_"*80)

# Rank salary within each department. Highest salary gets rank 1.
ranked_employees = employees.assign(
    dept_salary_rank=employees.groupby("department")["salary"].rank(method="dense", ascending=False)
)
print(ranked_employees)

   employee_id    name department       city  salary  experience_years
4          105   Elena         HR      Delhi   58000                 2
1          102     Ben      Sales     Mumbai   72000                 5
0          101    Asha      Sales      Delhi   65000                 3
3          104     Dev       Tech  Bengaluru  105000                 8
2          103  Chitra       Tech  Bengaluru   95000                 6
5          106   Farah       Tech  Hyderabad   88000                 4
________________________________________________________________________________
   employee_id    name department       city  salary  experience_years  dept_salary_rank
0          101    Asha      Sales      Delhi   65000                 3               2.0
1          102     Ben      Sales     Mumbai   72000                 5               1.0
2          103  Chitra       Tech  Bengaluru   95000                 6               2.0
3          104     Dev       Tech  Bengaluru  105000              

## 8. GroupBy: Split, Apply, Combine

`groupby` is one of the most important pandas tools.

The pattern is:

1. Split data into groups.
2. Apply aggregations or transformations.
3. Combine results into a new object.

Use named aggregation to produce clean column names.

In [22]:
department_summary = (
    employees
    .groupby("department")
    .agg(
        employees_count=("employee_id", "count"),
        avg_salary=("salary", "mean"),
        max_salary=("salary", "max"),
        avg_experience=("experience_years", "mean")
    )
    .round(2)
    .sort_values("avg_salary", ascending=False)
)

department_summary

,employees_count,avg_salary,max_salary,avg_experience
department,,,,
Tech,3,96000.0,105000,6.0
Sales,2,68500.0,72000,4.0
HR,1,58000.0,58000,2.0


## 9. Transform vs Aggregate

`agg()` reduces each group to fewer rows.

`transform()` returns a result aligned to the original rows.

Use `transform()` when you want group-level statistics beside row-level records.

In [24]:
employees

,employee_id,name,department,city,salary,experience_years
0,101,Asha,Sales,Delhi,65000,3
1,102,Ben,Sales,Mumbai,72000,5
2,103,Chitra,Tech,Bengaluru,95000,6
3,104,Dev,Tech,Bengaluru,105000,8
4,105,Elena,HR,Delhi,58000,2
5,106,Farah,Tech,Hyderabad,88000,4


In [23]:
employees_with_context = employees.assign(
    dept_avg_salary=employees.groupby("department")["salary"].transform("mean"),
)

# Compare each employee salary against their department average.
employees_with_context["salary_vs_dept_avg"] = (
    employees_with_context["salary"] - employees_with_context["dept_avg_salary"]
)

employees_with_context

,employee_id,name,department,city,salary,experience_years,dept_avg_salary,salary_vs_dept_avg
0,101,Asha,Sales,Delhi,65000,3,68500.0,-3500.0
1,102,Ben,Sales,Mumbai,72000,5,68500.0,3500.0
2,103,Chitra,Tech,Bengaluru,95000,6,96000.0,-1000.0
3,104,Dev,Tech,Bengaluru,105000,8,96000.0,9000.0
4,105,Elena,HR,Delhi,58000,2,58000.0,0.0
5,106,Farah,Tech,Hyderabad,88000,4,96000.0,-8000.0


## 10. Combining Data: Merge, Join, and Concat

Pandas supports SQL-style joins with `merge()`.

Common join types:

- `inner`: keep matching keys only.
- `left`: keep all rows from the left table.
- `right`: keep all rows from the right table.
- `outer`: keep all keys from both tables.

Use `validate` to protect yourself from accidental many-to-many joins.

In [27]:
department_targets = pd.DataFrame({
    "department": ["Sales", "Tech", "HR"],
    "quarterly_target": [500000, 900000, 220000]
})
print(department_targets)
print("_"*90)

# Many employee rows can match one department target row.
employees_with_targets = employees.merge(
    department_targets,
    on="department",
    how="left",
    validate="many_to_one"
)
print(employees_with_targets)

# Concatenate rows from another table with the same schema.
new_hires = pd.DataFrame({
    "employee_id": [107],
    "name": ["Gaurav"],
    "department": ["Sales"],
    "city": ["Jaipur"],
    "salary": [61000],
    "experience_years": [1]
})

all_employees = pd.concat([employees, new_hires], ignore_index=True)

all_employees

  department  quarterly_target
0      Sales            500000
1       Tech            900000
2         HR            220000
__________________________________________________________________________________________
   employee_id    name department       city  salary  experience_years  quarterly_target
0          101    Asha      Sales      Delhi   65000                 3            500000
1          102     Ben      Sales     Mumbai   72000                 5            500000
2          103  Chitra       Tech  Bengaluru   95000                 6            900000
3          104     Dev       Tech  Bengaluru  105000                 8            900000
4          105   Elena         HR      Delhi   58000                 2            220000
5          106   Farah       Tech  Hyderabad   88000                 4            900000


,employee_id,name,department,city,salary,experience_years
0,101,Asha,Sales,Delhi,65000,3
1,102,Ben,Sales,Mumbai,72000,5
2,103,Chitra,Tech,Bengaluru,95000,6
3,104,Dev,Tech,Bengaluru,105000,8
4,105,Elena,HR,Delhi,58000,2
5,106,Farah,Tech,Hyderabad,88000,4
6,107,Gaurav,Sales,Jaipur,61000,1


## 11. Reshaping Data

Many analytics tasks require changing between wide and long formats.

- `melt()` converts wide data to long data.
- `pivot()` converts long data to wide data.
- `pivot_table()` is like pivot plus aggregation.
- `stack()` and `unstack()` reshape index levels.

In [ ]:
monthly_sales = pd.DataFrame({
    "store": ["A", "B", "C"],
    "Jan": [120, 90, 140],
    "Feb": [135, 95, 150],
    "Mar": [128, 105, 160]
})

# Convert month columns into row values.
sales_long = monthly_sales.melt(
    id_vars="store",
    var_name="month",
    value_name="sales"
)

# Convert long format back to wide format.
sales_wide = sales_long.pivot(index="store", columns="month", values="sales").reset_index()

sales_long, sales_wide

## 12. Time Series with Pandas

Pandas has strong time series support.

Useful tools:

- `to_datetime()` converts strings to datetime dtype.
- `date_range()` creates regular date ranges.
- `resample()` groups datetime-indexed data by time frequency.
- `rolling()` calculates moving-window statistics.

In [ ]:
orders = pd.DataFrame({
    "order_date": [
        "2026-01-01", "2026-01-03", "2026-01-08", "2026-02-02",
        "2026-02-15", "2026-03-01", "2026-03-12"
    ],
    "revenue": [12000, 8500, 14200, 9900, 16300, 17500, 13200]
})

# Convert text dates to datetime values.
orders["order_date"] = pd.to_datetime(orders["order_date"])

# Set date as index so resample can group by calendar frequency.
orders_by_date = orders.set_index("order_date")

# Monthly revenue totals.
monthly_revenue = orders_by_date.resample("MS")["revenue"].sum()

# Rolling 2-order average based on row order.
orders["rolling_2_order_avg"] = orders["revenue"].rolling(window=2).mean()

monthly_revenue, orders

## 13. String Operations

Pandas provides vectorized string methods through `.str`.

These are useful for cleaning names, extracting codes, normalizing text, and building features.

In [ ]:
messy_products = pd.DataFrame({
    "sku": [" prd-001 ", "PRD-002", "prd-003 ", " service-004"],
    "label": ["Laptop Pro", "Laptop Air", "Phone Max", "Setup Service"]
})

products_clean = messy_products.assign(
    sku_clean=messy_products["sku"].str.strip().str.upper(),
    sku_prefix=messy_products["sku"].str.strip().str.split("-").str[0].str.upper(),
    is_product=messy_products["sku"].str.strip().str.startswith("prd")
)

products_clean

## 14. Categorical Data

Categorical dtype is useful when a column has repeated labels.

Benefits:

- Can reduce memory usage.
- Can define ordered categories.
- Makes category intent explicit.

In [ ]:
employees_categorical = employees_enriched.copy()

# Convert department to categorical dtype.
employees_categorical["department"] = employees_categorical["department"].astype("category")

# Define an ordered category for salary bands.
employees_categorical["salary_band"] = pd.Categorical(
    employees_categorical["salary_band"],
    categories=["low", "medium", "high"],
    ordered=True
)

employees_categorical.dtypes

## 15. Method Chaining

Method chaining keeps transformations readable by expressing a data pipeline top to bottom.

Helpful methods for chains:

- `assign()` creates columns.
- `query()` filters rows with readable expressions.
- `pipe()` passes a DataFrame into a custom function.
- `rename()` cleans column names.
- `sort_values()` orders the result.

In [ ]:
def add_salary_flags(df):
    """Add simple salary flags to a DataFrame."""
    return df.assign(
        above_company_avg=df["salary"] > df["salary"].mean(),
        salary_lakh=(df["salary"] / 100000).round(2)
    )


tech_salary_view = (
    employees
    .pipe(add_salary_flags)
    .query("department == 'Tech'")
    .sort_values("salary", ascending=False)
    .loc[:, ["name", "city", "salary", "salary_lakh", "above_company_avg"]]
)

tech_salary_view

## 16. Window Functions

Window functions compute values over neighboring rows.

Common examples:

- `rolling()` for moving averages.
- `expanding()` for cumulative calculations.
- `shift()` for lag or lead features.
- `pct_change()` for percentage change.

In [ ]:
traffic = pd.DataFrame({
    "date": pd.date_range("2026-01-01", periods=7, freq="D"),
    "visits": [100, 120, 115, 140, 160, 155, 180]
})

traffic_features = traffic.assign(
    visits_lag_1=traffic["visits"].shift(1),
    daily_change=traffic["visits"].diff(),
    pct_change=traffic["visits"].pct_change().round(3),
    rolling_3_day_avg=traffic["visits"].rolling(window=3).mean().round(2),
    cumulative_visits=traffic["visits"].expanding().sum()
)

traffic_features

## 17. Performance Notes

Good pandas performance usually comes from choosing the right operation.

Practical rules:

- Prefer vectorized operations over Python loops.
- Select only columns you need.
- Use categorical dtype for repeated strings.
- Avoid chained assignment; use `.loc` or `.assign()`.
- Use `copy()` intentionally when you need an independent DataFrame.
- Use `memory_usage(deep=True)` to inspect memory use.

In [ ]:
memory_before = employees.memory_usage(deep=True)

optimized = employees.copy()
optimized["department"] = optimized["department"].astype("category")
optimized["city"] = optimized["city"].astype("category")

memory_after = optimized.memory_usage(deep=True)

memory_comparison = pd.DataFrame({
    "before_bytes": memory_before,
    "after_bytes": memory_after,
    "saved_bytes": memory_before - memory_after
})

memory_comparison

## 18. Mini Project: Sales Analysis

This final section combines common pandas skills:

- Creating a realistic dataset.
- Cleaning date and text fields.
- Joining dimension data.
- Creating revenue and margin metrics.
- Aggregating by region and product category.
- Producing an executive summary table.

In [ ]:
transactions = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    "order_date": ["2026-01-05", "2026-01-08", "2026-01-20", "2026-02-02", "2026-02-14", "2026-02-22", "2026-03-01", "2026-03-08"],
    "customer_region": ["North", "South", "North", "West", "South", "West", "North", "South"],
    "product_id": ["P01", "P02", "P01", "P03", "P02", "P03", "P04", "P01"],
    "units": [3, 2, 5, 1, 4, 2, 6, 3],
    "unit_price": [1200, 1800, 1200, 2500, 1800, 2500, 900, 1200],
    "unit_cost": [800, 1200, 800, 1700, 1200, 1700, 500, 800]
})

products = pd.DataFrame({
    "product_id": ["P01", "P02", "P03", "P04"],
    "product_name": ["Starter Kit", "Pro Kit", "Enterprise Kit", "Accessory Pack"],
    "category": ["Core", "Core", "Enterprise", "Add-on"]
})

sales_analysis = (
    transactions
    .assign(order_date=lambda df: pd.to_datetime(df["order_date"]))
    .merge(products, on="product_id", how="left", validate="many_to_one")
    .assign(
        revenue=lambda df: df["units"] * df["unit_price"],
        cost=lambda df: df["units"] * df["unit_cost"],
        gross_profit=lambda df: df["revenue"] - df["cost"],
        month=lambda df: df["order_date"].dt.to_period("M").astype(str)
    )
)

executive_summary = (
    sales_analysis
    .groupby(["customer_region", "category"])
    .agg(
        orders=("order_id", "nunique"),
        units_sold=("units", "sum"),
        revenue=("revenue", "sum"),
        gross_profit=("gross_profit", "sum")
    )
    .assign(gross_margin_pct=lambda df: (df["gross_profit"] / df["revenue"] * 100).round(2))
    .sort_values("revenue", ascending=False)
)

executive_summary

## 19. Exporting Data

Pandas can export to many formats, including CSV, Excel, JSON, Parquet, and SQL databases.

Examples:

```python
df.to_csv("file.csv", index=False)
df.to_excel("file.xlsx", index=False)
df.to_json("file.json", orient="records")
```

In this notebook, export commands are shown as examples but not executed automatically, so the notebook stays clean.

## 20. Quick Revision Checklist

- Use `info()`, `describe()`, and `value_counts()` before changing data.
- Use `loc` and `iloc` for explicit selection.
- Use vectorized operations and `assign()` for new columns.
- Use `groupby().agg()` for summaries.
- Use `transform()` when group values must align back to original rows.
- Use `merge(..., validate=...)` to catch join mistakes.
- Use `melt()` and `pivot_table()` for reshaping.
- Use `to_datetime()`, `resample()`, and `rolling()` for time series.
- Use categorical dtype for repeated labels.
- Prefer readable method chains for multi-step analysis.